# Competencia — Aprendizaje de Máquina 2026-10
## Parte 2: Clasificación de Textos Históricos con Deep Learning

Este notebook implementa la Parte 2 del proyecto de clasificación de textos históricos
en español/latín según su **década de origen** (39 clases, de 1500 a 1880). El objetivo
es superar el Private Score de referencia de la Parte 1 (0.29144) usando arquitecturas
de deep learning y transferencia de aprendizaje.

El enfoque adopta una estrategia en dos niveles diseñada para entrenamiento en CPU:

- **Nivel 1 — MLP sobre TF-IDF:** red neuronal densa que reemplaza el LinearSVC de la
  Parte 1 sobre la misma representación TF-IDF. Cumple el requisito de arquitectura de
  deep learning y establece un punto de comparación interno directo con la Parte 1.

- **Nivel 2 — Transfer Learning con MiniLM:** modelo principal. Se usa
  `paraphrase-multilingual-MiniLM-L12-v2`, un transformer multilingüe destilado de
  BERT entrenado en más de 50 idiomas (incluyendo español y latín). Genera embeddings
  densos de 384 dimensiones que capturan semántica contextual imposible de representar
  con TF-IDF. Sobre estos embeddings se entrena una cabeza clasificadora MLP con las
  41 features lingüísticas de la Parte 1 concatenadas. Esta arquitectura satisface los
  requisitos de deep learning, transferencia de aprendizaje y es ejecutable en CPU en
  2-4 horas sobre el dataset completo.

**Métrica objetivo:** Accuracy (leaderboard Kaggle) y F1-macro (optimización interna).

In [1]:
import pandas as pd
import numpy as np
import re, os, math, warnings, random
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Reproducibilidad ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Constantes del experimento ──
BASELINE_P1_LOCAL   = 0.2737   # F1-macro del mejor modelo Parte 1 (80/20 local)
BASELINE_P1_KAGGLE  = 0.29144  # Private Score Kaggle Parte 1 — score a superar
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

os.makedirs('./submissions', exist_ok=True)
os.makedirs('./model', exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
print('✅ Listo')

PyTorch: 2.8.0
Device:  mps
✅ Listo


## 1. Carga de Datos

El dataset es idéntico al de la Parte 1:
- `train.csv`: 31.403 textos etiquetados con su década de origen
- `eval.csv`: 3.490 textos sin etiqueta sobre los cuales se generan las predicciones finales

La variable objetivo `decade` representa los tres primeros dígitos del año (ej: `164` para
la década de 1640), dando un total de **39 clases** desde 150 hasta 188.

In [2]:
df_train = pd.read_csv('./Data/train.csv')
df_eval  = pd.read_csv('./Data/eval.csv')

print(f'Train: {df_train.shape} | Eval: {df_eval.shape}')
print(f'Clases únicas: {df_train["decade"].nunique()}')
print(f'Distribución de clases (min/max textos): '
      f'{df_train["decade"].value_counts().min()} / '
      f'{df_train["decade"].value_counts().max()}')

Train: (31403, 2) | Eval: (3490, 2)
Clases únicas: 39
Distribución de clases (min/max textos): 754 / 848


El dataset contiene 31.403 textos de entrenamiento distribuidos en 39 clases con una
distribución casi uniforme — entre 754 y 848 textos por década. Esta uniformidad es
favorable para el entrenamiento: no hay sesgo por desbalance de clases y no se requiere
class weighting ni oversampling.

## 2. Preprocesamiento — Normalización OCR

Se reutiliza la función `normalize_ocr` de la Parte 1, que corrige artefactos tipográficos
y de digitalización sin tocar la ortografía arcaica del texto. El tokenizador de MiniLM
opera a nivel de subpalabra (WordPiece), por lo que formas como `hazer`, `dize` o `vna`
se tokenizarán en sus propios subwords — preservar esta señal es crítico para que el
modelo aprenda la evolución temporal del español.

Se eliminan además los 51 duplicados detectados en la Parte 1, quedando **31.352 textos**
para entrenamiento.

In [3]:
# ── Mapa de sustituciones OCR (idéntico a Parte 1) ──
CHAR_MAP = [
    ('\ufb01','fi'),('\ufb02','fl'),('\ufb00','ff'),('\ufb03','ffi'),('\ufb04','ffl'),
    ('\xe6','ae'),('\u0153','oe'),
    ('-\n',''),('- \n',''),('\xad',''),
    ('\xbb',' '),('\xab',' '),
    ('\u2018',"'"),("\u2019","'"),("\u201c",'"'),("\u201d",'"'),
    ('\xa3',' '),('\xa7',' '),('\xb6',' '),
    ('\u2020',' '),('\u2021',' '),('\u2022',' '),
    ('\u2014',' '),('\u2013',' '),
]

def normalize_ocr(text):
    text = str(text)
    for src, tgt in CHAR_MAP:
        text = text.replace(src, tgt)
    text = text.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    return re.sub(r'  +', ' ', text).strip()

# ── Aplicar normalización y eliminar duplicados ──
data = df_train.drop_duplicates(subset='text').reset_index(drop=True).copy()
data['text_clean']    = data['text'].apply(normalize_ocr)
df_eval['text_clean'] = df_eval['text'].apply(normalize_ocr)

print(f'Train sin duplicados: {len(data)}')
print(f'Duplicados eliminados: {len(df_train) - len(data)}')

Train sin duplicados: 31352
Duplicados eliminados: 51


Se confirmaron los 51 duplicados detectados en la Parte 1. El dataset queda en 31.352
textos limpios. La normalización OCR preserva la ortografía arcaica intacta — las
sustituciones solo corrigen artefactos de digitalización (ligaduras, guiones de corte
de línea, símbolos tipográficos) que no aportan señal temporal.

## 3. Extracción de Features Lingüísticas Históricas

Se reutilizan las **41 features numéricas** de la Parte 1, que capturan características
morfológicas, ortográficas y tipográficas correlacionadas con la época del texto. Estas
features se concatenarán más adelante a los embeddings de MiniLM antes de la capa
clasificadora, aportando señal explícita que el transformer puede no capturar
directamente desde el texto crudo — en particular los patrones regex de latín y
ortografía arcaica.

In [4]:
# ── Patrones regex históricos (idénticos a Parte 1) ──
RE_LONG_S    = re.compile(r'[bcdfghjklmnpqrstvwxyz]f[aeiouáéíóú]', re.I)
RE_V_AS_U    = re.compile(r'\bvn[aeiouáéíóú]|\bvn\b|\bvm\b', re.I)
RE_ROMAN     = re.compile(
    r'\b(M{1,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})'
    r'|CM|CD|XC|XL|IX|IV|D?C{2,3}|L?X{2,3}|V?I{2,4})\b')
RE_LATIN_END = re.compile(r'\b\w{3,}(orum|ibus|atis|endi|antis|entis)\b', re.I)
RE_LATIN_NOM = re.compile(r'\b\w{3,}(um|us|ae)\b', re.I)
RE_QU_ARC    = re.compile(r'\bqu[aou]\w', re.I)
RE_SS        = re.compile(r'ss', re.I)
RE_FF        = re.compile(r'ff', re.I)
RE_CION      = re.compile(r'\b\w{3,}cion\b', re.I)
RE_TION      = re.compile(r'\b\w{3,}tion\b', re.I)
RE_SION      = re.compile(r'\b\w{3,}sion\b', re.I)
RE_ABBREV    = re.compile(r'\b[A-Za-z]{1,4}\.')
RE_MCASE     = re.compile(r'\b[A-Z][a-z]{1,}[A-Z]\w*\b')
RE_RDIAC     = re.compile(r'[àâãäāăąèêëēĕěîïīĭôõōŏùûüūŭ]', re.I)
RE_SEMI      = re.compile(r';')
RE_COLON     = re.compile(r':')
RE_PAREN     = re.compile(r'[()]')

def extract_features(text):
    words   = text.split()
    n       = max(len(words), 1)
    nc      = max(len(text), 1)
    lengths = [len(w) for w in words]
    counts  = Counter(text)
    total   = len(text) or 1
    entropy = -sum((c/total)*math.log2(c/total) for c in counts.values()) if text else 0
    vowels  = sum(1 for c in text.lower() if c in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    cons    = sum(1 for c in text.lower() if c.isalpha()
                  and c not in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    sents   = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    ns      = max(len(sents), 1)
    return {
        'wl_mean':    np.mean(lengths) if lengths else 0,
        'wl_std':     np.std(lengths)  if lengths else 0,
        'wl_p75':     np.percentile(lengths, 75) if lengths else 0,
        'wl_p90':     np.percentile(lengths, 90) if lengths else 0,
        'ratio_long': sum(1 for l in lengths if l > 10) / n,
        'ratio_short':sum(1 for l in lengths if l <= 2) / n,
        'ratio_med':  sum(1 for l in lengths if 3 <= l <= 6) / n,
        'char_entropy':   entropy,
        'cv_ratio':       cons / vowels if vowels > 0 else 0,
        'word_char_ratio':n / nc,
        'comma_rate':     text.count(',') / n,
        'period_rate':    text.count('.') / n,
        'semicolon_rate': len(RE_SEMI.findall(text))  / n,
        'colon_rate':     len(RE_COLON.findall(text)) / n,
        'paren_rate':     len(RE_PAREN.findall(text)) / n,
        'excl_rate':      text.count('!') / n,
        'quest_rate':     text.count('?') / n,
        'total_punct':    sum(1 for c in text if c in '.,;:!?()[]{}') / n,
        'long_s_rate':    len(RE_LONG_S.findall(text))    / n,
        'v_as_u_rate':    len(RE_V_AS_U.findall(text))    / n,
        'latin_case':     len(RE_LATIN_END.findall(text)) / n,
        'latin_nom':      len(RE_LATIN_NOM.findall(text)) / n,
        'qu_archaic':     len(RE_QU_ARC.findall(text))    / n,
        'ss_rate':        len(RE_SS.findall(text))         / n,
        'ff_rate':        len(RE_FF.findall(text))         / n,
        'cion_rate':      len(RE_CION.findall(text))       / n,
        'tion_rate':      len(RE_TION.findall(text))       / n,
        'sion_rate':      len(RE_SION.findall(text))       / n,
        'abbrev_rate':    len(RE_ABBREV.findall(text))     / n,
        'roman_rate':     len(RE_ROMAN.findall(text))      / n,
        'rare_diac':      len(RE_RDIAC.findall(text))      / nc,
        'mixed_case':     len(RE_MCASE.findall(text))      / n,
        'ttr':         len(set(w.lower() for w in words)) / n,
        'upper_ratio': sum(1 for c in text if c.isupper()) / nc,
        'digit_ratio': sum(1 for c in text if c.isdigit()) / nc,
        'alpha_ratio': sum(1 for c in text if c.isalpha()) / nc,
        'all_caps':    sum(1 for w in words if w.isupper() and len(w)>1) / n,
        'pure_digit':  sum(1 for w in words if w.isdigit()) / n,
        'n_words':     float(n),
        'avg_sent_len':n / ns,
        'n_sentences': float(ns),
    }

print('Extrayendo features train...')
feats_train = pd.DataFrame(
    data['text_clean'].apply(extract_features).tolist()
).fillna(0)

print('Extrayendo features eval...')
feats_eval = pd.DataFrame(
    df_eval['text_clean'].apply(extract_features).tolist()
).fillna(0)

ALL_FEATS = feats_train.columns.tolist()
print(f'✅ Features extraídas: {len(ALL_FEATS)}')

Extrayendo features train...
Extrayendo features eval...
✅ Features extraídas: 41


Se extrajeron las 41 features lingüísticas históricas sobre los 31.352 textos de
entrenamiento y los 3.490 de evaluación. El conjunto incluye features de morfología
de palabras, estructura del texto, puntuación y patrones regex de latín y ortografía
arcaica — las mismas que en la Parte 1, donde demostraron aportar señal discriminativa
significativa sobre el TF-IDF solo.

## 4. Partición de Datos y Encoding de Etiquetas

Se realiza una partición estratificada 80/20 para evaluar los modelos localmente antes
de generar predicciones finales. El `LabelEncoder` convierte las 39 décadas (valores
enteros como 150, 151, ..., 188) a índices contiguos 0–38, formato requerido por
PyTorch para clasificación multiclase con `CrossEntropyLoss`.

In [5]:
# ── Encoding de etiquetas: décadas → índices 0-38 ──
le = LabelEncoder()
y_encoded = le.fit_transform(data['decade'].values)

print(f'Clases: {le.classes_[:5]} ... {le.classes_[-5:]}')
print(f'Índices: 0 ... {len(le.classes_) - 1}')

# ── Partición estratificada 80/20 ──
idx = np.arange(len(data))
idx_train, idx_val = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=y_encoded
)

y_train = y_encoded[idx_train]
y_val   = y_encoded[idx_val]

print(f'\nTrain: {len(idx_train)} textos | Val: {len(idx_val)} textos')
print(f'Clases en train: {len(np.unique(y_train))} | '
      f'Clases en val: {len(np.unique(y_val))}')

Clases: [150 151 152 153 154] ... [184 185 186 187 188]
Índices: 0 ... 38

Train: 25081 textos | Val: 6271 textos
Clases en train: 39 | Clases en val: 39


La partición estratificada garantiza representación de las 39 décadas en train y
validación. El LabelEncoder mapea las décadas originales (150–188) a índices 0–38
— la transformación inversa `le.inverse_transform()` se usará al final para recuperar
las décadas reales en el archivo de submission.

## 5. Nivel 1 — MLP sobre TF-IDF

Antes de pasar al modelo principal de transfer learning, se entrena una red neuronal
densa (MLP) sobre la misma representación TF-IDF de la Parte 1. Esto cumple el
requisito de arquitectura de deep learning y establece un punto de comparación interno
directo: si el MLP supera el F1-macro de 0.2737 del LinearSVC de la Parte 1 sobre el
mismo split 80/20, confirma que la arquitectura neuronal agrega valor incluso sin
embeddings contextuales.

La arquitectura usa dos capas ocultas con ReLU y Dropout, entrada de dimensión igual
al número de features TF-IDF + 41 features numéricas, y salida de 39 clases.
Dado que la matriz TF-IDF es sparse, se convierte a tensor denso por batches dentro
del DataLoader para no materializar toda la matriz en memoria de una vez.

In [6]:
# ── Vectorizadores TF-IDF (misma config ganadora de Parte 1) ──
print('Construyendo matrices TF-IDF...')

vec_char = TfidfVectorizer(
    analyzer='char', ngram_range=(1, 4),
    sublinear_tf=True, max_features=400_000,
    min_df=1, max_df=0.98, strip_accents=None,
)
vec_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    sublinear_tf=True, max_features=100_000,
    min_df=1, max_df=0.98, strip_accents=None,
)

texts = data['text_clean'].values

# fit sobre train, transform sobre val y eval
char_full = vec_char.fit_transform(texts)
word_full = vec_word.fit_transform(texts)

char_eval = vec_char.transform(df_eval['text_clean'].values)
word_eval = vec_word.transform(df_eval['text_clean'].values)

# ── Features numéricas escaladas ──
sc = StandardScaler()
num_full = sc.fit_transform(feats_train.values)
num_eval = sc.transform(feats_eval.values)

# ── Matrices combinadas (sparse + numérico) ──
X_full_sparse = hstack([char_full, word_full, csr_matrix(num_full)])
X_eval_sparse = hstack([char_eval, word_eval, csr_matrix(num_eval)])

# ── Splits ──
X_train_sparse = X_full_sparse[idx_train]
X_val_sparse   = X_full_sparse[idx_val]

print(f'Shape matriz completa: {X_full_sparse.shape}')
print(f'Shape train:           {X_train_sparse.shape}')
print(f'Shape val:             {X_val_sparse.shape}')
print('✅ Matrices TF-IDF listas')

Construyendo matrices TF-IDF...
Shape matriz completa: (31352, 500041)
Shape train:           (25081, 500041)
Shape val:             (6271, 500041)
✅ Matrices TF-IDF listas


Las matrices TF-IDF replican exactamente la representación ganadora de la Parte 1:
400k features de char n-gramas (1,4) + 100k de word bigramas + 41 features numéricas
= 500.041 dimensiones. Esta consistencia es intencional — cualquier diferencia de
score entre el LinearSVC de la Parte 1 y el MLP de la Parte 2 se debe exclusivamente
a la arquitectura del clasificador, no a la representación.

### 5.1 Dataset, Arquitectura y Entrenamiento del MLP

Se entrena una red neuronal densa (MLP) sobre embeddings generados por
`paraphrase-multilingual-MiniLM-L12-v2`, un transformer multilingüe destilado
entrenado en más de 50 idiomas incluyendo español y latín. MiniLM genera un
embedding denso de **384 dimensiones** por texto — una representación contextual
compacta que captura semántica imposible de representar con TF-IDF.

El pipeline es:
1. MiniLM codifica cada texto a un vector de 384 dims (transferencia de aprendizaje)
2. Se concatenan las 41 features lingüísticas → vector de 425 dims por texto
3. Un MLP de dos capas ocultas clasifica en 39 décadas

Esta arquitectura satisface todos los requisitos de la Parte 2: usa deep learning,
transferencia de aprendizaje con modelo preentrenado, y es ejecutable en CPU/MPS
en menos de una hora sobre el dataset completo.

In [11]:
from sentence_transformers import SentenceTransformer

# ── Cargar modelo preentrenado ──
print('Cargando MiniLM...')
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ── Generar embeddings — el paso más largo (~20-30 min en CPU/MPS) ──
print('Generando embeddings train...')
emb_train_full = embedder.encode(
    data['text_clean'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    device='mps',
)

print('Generando embeddings eval...')
emb_eval = embedder.encode(
    df_eval['text_clean'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    device='mps',
)

print(f'Shape embeddings train: {emb_train_full.shape}')
print(f'Shape embeddings eval:  {emb_eval.shape}')

# ── Guardar en disco para no regenerar si el kernel se reinicia ──
np.save('./model/emb_train_full.npy', emb_train_full)
np.save('./model/emb_eval.npy',       emb_eval)
print('✅ Embeddings guardados en ./model/')

Cargando MiniLM...
Generando embeddings train...


Batches: 100%|██████████| 490/490 [02:35<00:00,  3.15it/s]


Generando embeddings eval...


Batches: 100%|██████████| 55/55 [00:17<00:00,  3.20it/s]

Shape embeddings train: (31352, 384)
Shape embeddings eval:  (3490, 384)
✅ Embeddings guardados en ./model/


In [13]:
# ── Cargar desde disco si ya existen (evita regenerar) ──
# emb_train_full = np.load('./model/emb_train_full.npy')
# emb_eval       = np.load('./model/emb_eval.npy')

# ── Escalar features numéricas ──
sc = StandardScaler()
num_full_scaled = sc.fit_transform(feats_train.values)
num_eval_scaled = sc.transform(feats_eval.values)

# ── Concatenar embeddings + features lingüísticas ──
X_full = np.hstack([emb_train_full, num_full_scaled])  # (31352, 425)
X_eval = np.hstack([emb_eval,       num_eval_scaled])

X_tr = X_full[idx_train]
X_vl = X_full[idx_val]

print(f'Shape X_train: {X_tr.shape}')  # (25081, 425)
print(f'Shape X_val:   {X_vl.shape}')  # (6271, 425)
print(f'Shape X_eval:  {X_eval.shape}')

# ── Dataset y DataLoaders ──
class DenseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    DenseDataset(X_tr, y_train),
    batch_size=256, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    DenseDataset(X_vl, y_val),
    batch_size=256, shuffle=False, num_workers=0
)
print('✅ DataLoaders listos')

Shape X_train: (25081, 425)
Shape X_val:   (6271, 425)
Shape X_eval:  (3490, 425)
✅ DataLoaders listos


In [15]:
# ── Arquitectura MLP sobre embeddings 425-dim ──
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=425, hidden1=512, hidden2=256, n_classes=39):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, n_classes),
        )

    def forward(self, x):
        return self.net(x)

# ── Modelo ──
INPUT_DIM = X_tr.shape[1]
mlp = MLPClassifier(input_dim=INPUT_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

print(f'Parámetros: {sum(p.numel() for p in mlp.parameters()):,}')
print(f'Input dim:  {INPUT_DIM} | Device: {DEVICE}')

# ── Entrenamiento ──
EPOCHS = 30
history_mlp = {'train_loss': [], 'val_f1': [], 'val_acc': []}
best_f1, best_epoch = 0, 0

for epoch in range(1, EPOCHS + 1):
    mlp.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    mlp.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            preds = mlp(xb.to(DEVICE)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    val_f1  = f1_score(y_val, all_preds, average='macro')
    val_acc = accuracy_score(y_val, all_preds)
    avg_loss = total_loss / len(train_loader)

    history_mlp['train_loss'].append(avg_loss)
    history_mlp['val_f1'].append(val_f1)
    history_mlp['val_acc'].append(val_acc)
    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1, best_epoch = val_f1, epoch
        torch.save(mlp.state_dict(), './model/mlp_minilm_best.pt')

    mark = '✅' if val_f1 > BASELINE_P1_LOCAL else '  '
    print(f'Época {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | '
          f'Val F1: {val_f1:.4f} | Acc: {val_acc:.4f} {mark}')

print(f'\nMejor Val F1: {best_f1:.4f} (época {best_epoch})')

Parámetros: 360,999
Input dim:  425 | Device: mps
Época  1/30 | Loss: 3.1992 | Val F1: 0.1331 | Acc: 0.1555   
Época  2/30 | Loss: 2.8558 | Val F1: 0.1562 | Acc: 0.1741   
Época  3/30 | Loss: 2.7496 | Val F1: 0.1727 | Acc: 0.1832   
Época  4/30 | Loss: 2.6725 | Val F1: 0.1753 | Acc: 0.1899   
Época  5/30 | Loss: 2.6198 | Val F1: 0.1821 | Acc: 0.1936   
Época  6/30 | Loss: 2.5575 | Val F1: 0.1881 | Acc: 0.1968   
Época  7/30 | Loss: 2.5142 | Val F1: 0.1942 | Acc: 0.2057   
Época  8/30 | Loss: 2.4647 | Val F1: 0.1931 | Acc: 0.2028   
Época  9/30 | Loss: 2.4252 | Val F1: 0.1932 | Acc: 0.2024   
Época 10/30 | Loss: 2.3777 | Val F1: 0.1897 | Acc: 0.2020   
Época 11/30 | Loss: 2.3402 | Val F1: 0.1964 | Acc: 0.2036   
Época 12/30 | Loss: 2.3037 | Val F1: 0.1967 | Acc: 0.2019   
Época 13/30 | Loss: 2.2593 | Val F1: 0.1970 | Acc: 0.2017   
Época 14/30 | Loss: 2.2299 | Val F1: 0.1923 | Acc: 0.1982   
Época 15/30 | Loss: 2.1804 | Val F1: 0.1965 | Acc: 0.2027   
Época 16/30 | Loss: 2.1458 | Val F1

### 5.2 Predicciones Baseline — MLP sobre embeddings MiniLM

Se generan predicciones sobre `eval.csv` con el mejor checkpoint del MLP
(época 19, Val F1: 0.2001) como registro intermedio antes de continuar
con el fine-tuning. Este submission establece un piso de comparación interno.

In [16]:
# ── Cargar mejor checkpoint ──
mlp.load_state_dict(torch.load('./model/mlp_minilm_best.pt'))
mlp.eval()

# ── Predicciones sobre eval ──
X_eval_tensor = torch.tensor(X_eval, dtype=torch.float32)
eval_dataset  = DenseDataset(X_eval, np.zeros(len(X_eval), dtype=int))
eval_loader   = DataLoader(eval_dataset, batch_size=256, shuffle=False)

all_preds = []
with torch.no_grad():
    for xb, _ in eval_loader:
        preds = mlp(xb.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# ── Convertir índices → décadas reales ──
decades_pred = le.inverse_transform(all_preds)

# ── Guardar submission ──
sub = pd.DataFrame({'id': df_eval['id'], 'decade': decades_pred})
sub.to_csv('./submissions/SUBMISSION_mlp_minilm.csv', index=False)

print(f'Predicciones generadas: {len(sub)}')
print(f'Distribución de clases predichas:')
print(sub['decade'].value_counts().sort_index().to_string())
print('✅ Guardado en ./submissions/SUBMISSION_mlp_minilm.csv')

Predicciones generadas: 3490
Distribución de clases predichas:
decade
150     83
151    112
152    104
153     98
154     94
155     80
156    115
157     99
158    128
159     71
160     64
161     24
162     93
163     74
164    155
165     35
166     71
167     95
168     69
169     76
170    128
171     89
172    112
173    159
174     79
175     75
176     56
177     94
178     63
179     97
180     85
181     86
182    118
183     70
184     61
185     47
186    151
187    112
188     68
✅ Guardado en ./submissions/SUBMISSION_mlp_minilm.csv


## 6. Modelo Principal — Fine-tuning de MiniLM

El MLP sobre embeddings fijos de MiniLM alcanzó Val F1: 0.2001 — por debajo del
baseline de Parte 1 (0.2737). El problema es que los embeddings genéricos no están
especializados en distinguir décadas históricas.

La solución es **fine-tuning end-to-end**: entrenar los pesos del transformer junto
con la cabeza clasificadora para que las representaciones internas de MiniLM se
adapten específicamente a la tarea. Esto es transfer learning real — se aprovecha
el conocimiento lingüístico preentrenado del modelo y se especializa para el dominio.

El pipeline es:
- Tokenización con el tokenizador nativo de MiniLM (WordPiece — maneja ortografía arcaica)
- Encoder transformer completo fine-tuned
- Pooling del token [CLS] → vector de 384 dims
- Concatenación con 41 features lingüísticas → 425 dims
- Capa densa 256 → Dropout(0.3) → 39 clases

Se usa batch size 32 y gradient accumulation de 2 pasos para simular batch 64
sin exceder la memoria disponible. Learning rate 2e-5 con warmup — estándar para
fine-tuning de transformers.

In [7]:
from transformers import AutoTokenizer, AutoModel

# ── Tokenizador ──
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128

# ── Dataset con tokenización ──
class HistoricalTextDataset(Dataset):
    def __init__(self, texts, labels, num_features, tokenizer, max_len=128):
        self.texts        = texts
        self.labels       = torch.tensor(labels, dtype=torch.long)
        self.num_features = torch.tensor(num_features, dtype=torch.float32)
        self.tokenizer    = tokenizer
        self.max_len      = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'num_features':   self.num_features[idx],
            'label':          self.labels[idx],
        }

# ── Escalar features numéricas ──
sc2 = StandardScaler()
num_tr_scaled = sc2.fit_transform(feats_train.values[idx_train])
num_vl_scaled = sc2.transform(feats_train.values[idx_val])
num_ev_scaled = sc2.transform(feats_eval.values)

# ── Textos ──
texts_train = data['text_clean'].values[idx_train].tolist()
texts_val   = data['text_clean'].values[idx_val].tolist()
texts_eval  = df_eval['text_clean'].tolist()

# ── Datasets ──
train_dataset = HistoricalTextDataset(
    texts_train, y_train, num_tr_scaled, tokenizer, MAX_LEN
)
val_dataset = HistoricalTextDataset(
    texts_val, y_val, num_vl_scaled, tokenizer, MAX_LEN
)

# ── DataLoaders ──
ft_train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0
)
ft_val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=0
)

print(f'Train batches: {len(ft_train_loader)}')
print(f'Val batches:   {len(ft_val_loader)}')
print(f'Max tokens:    {MAX_LEN}')
print('✅ Datasets listos')

Train batches: 784
Val batches:   196
Max tokens:    128
✅ Datasets listos


In [8]:
class MiniLMClassifier(nn.Module):
    def __init__(self, model_name, num_features=41, n_classes=39, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size  # 384

        # ── Congelar todas las capas del encoder ──
        for param in self.encoder.parameters():
            param.requires_grad = False

        # ── Descongelar solo las últimas 2 capas transformer + layer norm final ──
        for layer in self.encoder.encoder.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        for param in self.encoder.pooler.parameters():
            param.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, input_ids, attention_mask, num_features):
        out     = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]
        x       = torch.cat([cls_emb, num_features], dim=1)
        return self.classifier(x)

# ── Instanciar modelo ──
ft_model = MiniLMClassifier(MODEL_NAME).to(DEVICE)

total_params     = sum(p.numel() for p in ft_model.parameters())
trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Parámetros totales:     {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}')
print(f'Parámetros congelados:  {frozen_params:,}')

Parámetros totales:     117,773,351
Parámetros entrenables: 3,816,359
Parámetros congelados:  113,956,992


In [9]:
from transformers import get_linear_schedule_with_warmup

os.makedirs('./model_finetuned', exist_ok=True)

EPOCHS_FT   = 5
ACCUM_STEPS = 2
LR_ENCODER  = 2e-5
LR_HEAD     = 1e-3

optimizer_ft = optim.AdamW([
    {'params': [p for p in ft_model.encoder.parameters() if p.requires_grad],
     'lr': LR_ENCODER},
    {'params': ft_model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=0.01)

total_steps  = (len(ft_train_loader) // ACCUM_STEPS) * EPOCHS_FT
warmup_steps = total_steps // 10

scheduler_ft = get_linear_schedule_with_warmup(
    optimizer_ft,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion_ft = nn.CrossEntropyLoss()

print(f'Total steps:  {total_steps}')
print(f'Warmup steps: {warmup_steps}')
print(f'LR encoder:   {LR_ENCODER} | LR cabeza: {LR_HEAD}')

# ── Entrenamiento ──
history_ft = {'train_loss': [], 'val_f1': [], 'val_acc': []}
best_f1_ft, best_epoch_ft = 0, 0

for epoch in range(1, EPOCHS_FT + 1):
    ft_model.train()
    total_loss = 0
    optimizer_ft.zero_grad()

    for step, batch in enumerate(tqdm(ft_train_loader,
                                      desc=f'Época {epoch}/{EPOCHS_FT}')):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        num_feats      = batch['num_features'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        logits = ft_model(input_ids, attention_mask, num_feats)
        loss   = criterion_ft(logits, labels) / ACCUM_STEPS
        loss.backward()

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
            optimizer_ft.step()
            scheduler_ft.step()
            optimizer_ft.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    # ── Validación ──
    ft_model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in ft_val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            num_feats      = batch['num_features'].to(DEVICE)
            preds = ft_model(
                input_ids, attention_mask, num_feats
            ).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    val_f1  = f1_score(y_val, all_preds, average='macro')
    val_acc = accuracy_score(y_val, all_preds)
    avg_loss = total_loss / len(ft_train_loader)

    history_ft['train_loss'].append(avg_loss)
    history_ft['val_f1'].append(val_f1)
    history_ft['val_acc'].append(val_acc)

    if val_f1 > best_f1_ft:
        best_f1_ft, best_epoch_ft = val_f1, epoch
        torch.save(ft_model.state_dict(),
                   './model_finetuned/minilm_finetuned_best.pt')

    mark = '✅' if val_f1 > BASELINE_P1_LOCAL else '  '
    print(f'Época {epoch}/{EPOCHS_FT} | Loss: {avg_loss:.4f} | '
          f'Val F1: {val_f1:.4f} | Acc: {val_acc:.4f} {mark}')

print(f'\nMejor Val F1: {best_f1_ft:.4f} (época {best_epoch_ft})')

Total steps:  1960
Warmup steps: 196
LR encoder:   2e-05 | LR cabeza: 0.001


Época 1/5: 100%|██████████| 784/784 [05:33<00:00,  2.35it/s]


Época 1/5 | Loss: 3.2727 | Val F1: 0.1368 | Acc: 0.1609   


Época 2/5: 100%|██████████| 784/784 [05:29<00:00,  2.38it/s]


Época 2/5 | Loss: 2.8643 | Val F1: 0.1639 | Acc: 0.1807   


...

Época 3/5 | Loss: 2.7526 | Val F1: 0.1747 | Acc: 0.1863   


Época 4/5: 100%|██████████| 784/784 [05:29<00:00,  2.38it/s]


Época 4/5 | Loss: 2.6721 | Val F1: 0.1820 | Acc: 0.1961   


Época 5/5: 100%|██████████| 784/784 [05:30<00:00,  2.37it/s]


Época 5/5 | Loss: 2.6277 | Val F1: 0.1889 | Acc: 0.1990   

Mejor Val F1: 0.1889 (época 5)


### 6.4 Predicciones Intermedias — Fine-tuning época 5

Se generan predicciones con el mejor checkpoint hasta ahora (época 5, Val F1: 0.1889)
como registro intermedio antes de reentrenar con más épocas.

In [10]:
# ── Dataset eval ──
eval_dataset_ft = HistoricalTextDataset(
    texts_eval,
    np.zeros(len(texts_eval), dtype=int),
    num_ev_scaled,
    tokenizer,
    MAX_LEN,
)
eval_loader_ft = DataLoader(
    eval_dataset_ft, batch_size=32, shuffle=False, num_workers=0
)

# ── Cargar mejor checkpoint ──
ft_model.load_state_dict(
    torch.load('./model_finetuned/minilm_finetuned_best.pt')
)
ft_model.eval()

# ── Predicciones ──
all_preds = []
with torch.no_grad():
    for batch in tqdm(eval_loader_ft, desc='Predicciones'):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        num_feats      = batch['num_features'].to(DEVICE)
        preds = ft_model(
            input_ids, attention_mask, num_feats
        ).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# ── Guardar submission ──
decades_pred = le.inverse_transform(all_preds)
sub = pd.DataFrame({'id': df_eval['id'], 'decade': decades_pred})
sub.to_csv('./submissions/SUBMISSION_minilm_ft_e5.csv', index=False)

print(f'Predicciones generadas: {len(sub)}')
print('✅ Guardado en ./submissions/SUBMISSION_minilm_ft_e5.csv')

Predicciones: 100%|██████████| 110/110 [00:23<00:00,  4.70it/s]

Predicciones generadas: 3490
✅ Guardado en ./submissions/SUBMISSION_minilm_ft_e5.csv


### 6.5 Reentrenamiento — 10 épocas

El modelo alcanzó Val F1: 0.1889 en 5 épocas con tendencia creciente sin señales
de plateau — la loss bajó de 3.27 a 2.63 de forma consistente. Se reentrana desde
cero con 10 épocas para permitir convergencia completa. Se espera superar el
baseline de Parte 1 (0.2737) alrededor de la época 7-8. El checkpoint se guarda
en una ruta separada para no sobreescribir el modelo de 5 épocas.

In [11]:
# ── Reiniciar modelo desde cero ──
ft_model = MiniLMClassifier(MODEL_NAME).to(DEVICE)

EPOCHS_FT   = 10
ACCUM_STEPS = 2
LR_ENCODER  = 2e-5
LR_HEAD     = 1e-3

optimizer_ft = optim.AdamW([
    {'params': [p for p in ft_model.encoder.parameters() if p.requires_grad],
     'lr': LR_ENCODER},
    {'params': ft_model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=0.01)

total_steps  = (len(ft_train_loader) // ACCUM_STEPS) * EPOCHS_FT
warmup_steps = total_steps // 10

scheduler_ft = get_linear_schedule_with_warmup(
    optimizer_ft,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion_ft = nn.CrossEntropyLoss()

print(f'Total steps:  {total_steps}')
print(f'Warmup steps: {warmup_steps}')
print(f'LR encoder:   {LR_ENCODER} | LR cabeza: {LR_HEAD}')

# ── Entrenamiento ──
history_ft = {'train_loss': [], 'val_f1': [], 'val_acc': []}
best_f1_ft, best_epoch_ft = 0, 0

for epoch in range(1, EPOCHS_FT + 1):
    ft_model.train()
    total_loss = 0
    optimizer_ft.zero_grad()

    for step, batch in enumerate(tqdm(ft_train_loader,
                                      desc=f'Época {epoch}/{EPOCHS_FT}')):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        num_feats      = batch['num_features'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        logits = ft_model(input_ids, attention_mask, num_feats)
        loss   = criterion_ft(logits, labels) / ACCUM_STEPS
        loss.backward()

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
            optimizer_ft.step()
            scheduler_ft.step()
            optimizer_ft.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    # ── Validación ──
    ft_model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in ft_val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            num_feats      = batch['num_features'].to(DEVICE)
            preds = ft_model(
                input_ids, attention_mask, num_feats
            ).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    val_f1  = f1_score(y_val, all_preds, average='macro')
    val_acc = accuracy_score(y_val, all_preds)
    avg_loss = total_loss / len(ft_train_loader)

    history_ft['train_loss'].append(avg_loss)
    history_ft['val_f1'].append(val_f1)
    history_ft['val_acc'].append(val_acc)

    if val_f1 > best_f1_ft:
        best_f1_ft, best_epoch_ft = val_f1, epoch
        torch.save(ft_model.state_dict(),
                   './model_finetuned/minilm_finetuned_10e_best.pt')

    mark = '✅' if val_f1 > BASELINE_P1_LOCAL else '  '
    print(f'Época {epoch:2d}/{EPOCHS_FT} | Loss: {avg_loss:.4f} | '
          f'Val F1: {val_f1:.4f} | Acc: {val_acc:.4f} {mark}')

print(f'\nMejor Val F1: {best_f1_ft:.4f} (época {best_epoch_ft})')

Total steps:  3920
Warmup steps: 392
LR encoder:   2e-05 | LR cabeza: 0.001


Época 1/10: 100%|██████████| 784/784 [05:34<00:00,  2.34it/s]


Época  1/10 | Loss: 3.3612 | Val F1: 0.1371 | Acc: 0.1607   


Época 2/10: 100%|██████████| 784/784 [05:30<00:00,  2.37it/s]


Época  2/10 | Loss: 2.9003 | Val F1: 0.1672 | Acc: 0.1847   


...

Época  8/10 | Loss: 2.5013 | Val F1: 0.2014 | Acc: 0.2087   


Época 9/10: 100%|██████████| 784/784 [05:33<00:00,  2.35it/s]


Época  9/10 | Loss: 2.4731 | Val F1: 0.1992 | Acc: 0.2089   


Época 10/10: 100%|██████████| 784/784 [05:30<00:00,  2.37it/s]


Época 10/10 | Loss: 2.4510 | Val F1: 0.2038 | Acc: 0.2110   

Mejor Val F1: 0.2038 (época 10)


### 6.6 Predicciones — Fine-tuning 10 épocas

Se generan predicciones sobre `eval.csv` con el mejor checkpoint del reentrenamiento
(época 10, Val F1: 0.2038).

In [12]:
# ── Dataset eval ──
eval_dataset_ft = HistoricalTextDataset(
    texts_eval,
    np.zeros(len(texts_eval), dtype=int),
    num_ev_scaled,
    tokenizer,
    MAX_LEN,
)
eval_loader_ft = DataLoader(
    eval_dataset_ft, batch_size=32, shuffle=False, num_workers=0
)

# ── Cargar mejor checkpoint ──
ft_model.load_state_dict(
    torch.load('./model_finetuned/minilm_finetuned_10e_best.pt')
)
ft_model.eval()

# ── Predicciones ──
all_preds = []
with torch.no_grad():
    for batch in tqdm(eval_loader_ft, desc='Predicciones'):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        num_feats      = batch['num_features'].to(DEVICE)
        preds = ft_model(
            input_ids, attention_mask, num_feats
        ).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# ── Guardar submission ──
decades_pred = le.inverse_transform(all_preds)
sub = pd.DataFrame({'id': df_eval['id'], 'decade': decades_pred})
sub.to_csv('./submissions/SUBMISSION_minilm_ft_10e.csv', index=False)

print(f'Predicciones generadas: {len(sub)}')
print('✅ Guardado en ./submissions/SUBMISSION_minilm_ft_10e.csv')

Predicciones: 100%|██████████| 110/110 [00:22<00:00,  4.80it/s]


Predicciones generadas: 3490
✅ Guardado en ./submissions/SUBMISSION_minilm_ft_10e.csv
